# Data audit, cleaning and splitting

**Decision supported:** establish a traceable dataset for a retrospective retention study. Input: the supplied raw CSV. Internal outputs: a dataset without business identifiers, immutable split assignments and a data contract. No learning or exploration of predictor–target relationships takes place here.

Run notebooks 01 → 04 with `.venv/bin/python src/run_study.py`, or open `.venv/bin/jupyter lab notebooks/`. Each step starts in a fresh kernel; all handoffs go through `data/`. Public exports contain aggregates only.

The notebooks are in English as requested. Legacy artifact keys and sealed protocol descriptions retain their original spelling to preserve the study's audit trail; displayed labels are translated without changing the data.

In [1]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
ROOT = Path.cwd() if (Path.cwd() / 'brief.md').exists() else Path.cwd().parent
assert (ROOT / 'brief.md').exists(), 'Run from the project root or notebooks/'
sys.path.insert(0, str(ROOT / 'src'))
from presentation import display, en
SEED = 42
OUT = ROOT / 'data/results'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()

## Documented target and scope of verification

[Microsoft Learn — bank churn example](https://learn.microsoft.com/en-us/fabric/data-science/customer-churn) (accessed November 9, 2026) defines churn as account closure. Its table has the same structure and first two observations as the supplied file, and reports matching class counts. We therefore **infer** the educational dataset's convention: `Exited=1` means departure/account closure; `0` means no observed departure. The encoding is checked below without constructing a proxy.

This match does not establish complete file identity, original provenance, licensing or bank approval. The study population is only the supplied sample of customers in three countries; its representativeness is unknown. The observation unit is one row per customer, subject to the uniqueness check.

**Unknowns preventing deployment:** snapshot date, collection period, decision date, label horizon, time to closure, possible reopening, censoring, availability of each feature before the event, currency and the precise definition of activity. Age is interpreted in years; the external documentation describes tenure in years. No timeline is reconstructed. Models will predict the supplied label, not a 12-month churn probability. Business owners must confirm these points on a dated cohort before use.

In [2]:
raw_path = ROOT / 'data/raw/Churn_Modelling.csv'
raw_hash = sha(raw_path)
df = pd.read_csv(raw_path)
expected = ['RowNumber','CustomerId','Surname','CreditScore','Geography','Gender','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary','Exited']
assert df.columns.tolist() == expected
assert df.shape == (10000, 14)
assert set(df.Exited.unique()) == {0, 1}
audit = pd.DataFrame({'type': df.dtypes.astype(str), 'manquants': df.isna().sum(), 'modalités': df.nunique()})
display(audit)
display(df[['Exited']].value_counts().rename('effectif').to_frame())
quality = {'lignes': len(df), 'colonnes': len(df.columns), 'manquants': int(df.isna().sum().sum()), 'doublons_exacts': int(df.duplicated().sum()), 'clients_repetes': int(df.CustomerId.duplicated().sum()), 'numeros_repetes': int(df.RowNumber.duplicated().sum())}
display(pd.Series(quality, name='audit'))
assert quality['clients_repetes'] == quality['doublons_exacts'] == quality['manquants'] == 0
assert df.CustomerId.gt(0).all() and df.RowNumber.tolist() == list(range(1, len(df)+1))

,Data type,Missing,Unique values
RowNumber,int64,0,10000
CustomerId,int64,0,10000
Surname,str,0,2932
CreditScore,int64,0,460
Geography,str,0,3
Gender,str,0,2
Age,int64,0,70
Tenure,int64,0,11
Balance,float64,0,6382
NumOfProducts,int64,0,4


,Count
Exited,
0,7963
1,2037


,Audit
Rows,10000
Columns,14
Missing,0
Exact duplicates,0
Repeated customer IDs,0
Repeated row numbers,0


## Deterministic structural checks

The following bounds detect obvious inconsistencies without learning thresholds from the test set: credit score between 0 and 1,000 (exact scale to be confirmed), age between 18 and 120 years, non-negative tenure no greater than age, non-negative amounts, positive integer product count, and binary indicators. Country and gender categories are checked against this file's observed categories, without claiming to cover every possible population. Any failure stops execution; there are no silent corrections.

A zero balance can be genuine. Older age or three/four products is not a reason to drop a customer. No capping, quantile filtering, imputation, learned encoding or scaling is applied here. Full descriptive statistics are reserved for the training set in notebook 02.

In [3]:
checks = {
 'score_domaine_large': df.CreditScore.between(0,1000),
 'age_adulte_plausible': df.Age.between(18,120),
 'anciennete_plausible': df.Tenure.ge(0) & df.Tenure.le(df.Age),
 'solde_non_negatif': df.Balance.ge(0),
 'salaire_non_negatif': df.EstimatedSalary.ge(0),
 'produits_entiers_positifs': df.NumOfProducts.ge(1) & df.NumOfProducts.mod(1).eq(0),
 'carte_binaire': df.HasCrCard.isin([0,1]),
 'activite_binaire': df.IsActiveMember.isin([0,1]),
 'pays_attendus': df.Geography.isin(['France','Spain','Germany']),
 'genre_observe': df.Gender.isin(['Female','Male']),
}
for c in ['Geography','Gender','Surname']:
 checks['texte_sans_espaces_'+c] = df[c].eq(df[c].str.strip()) & df[c].str.len().gt(0)
for c in df.select_dtypes(include='number'):
 checks['fini_'+c] = pd.Series(np.isfinite(df[c]), index=df.index)
violations = pd.Series({k: int((~v).sum()) for k,v in checks.items()}, name='violations')
display(violations.to_frame())
assert violations.sum() == 0
clean = df.drop(columns=['RowNumber','CustomerId','Surname']).copy()
clean.insert(0, 'observation_id', np.arange(len(df)))
display(pd.DataFrame({'brut': [len(df),len(df.columns),0], 'préparé': [len(clean),len(clean.columns),0]}, index=['lignes','colonnes (clé technique incluse)','valeurs modifiées']))

,violations
Credit score within broad range,0
Plausible adult age,0
Plausible tenure,0
Non-negative balance,0
Non-negative salary,0
Positive integer product count,0
Binary card indicator,0
Binary activity indicator,0
Expected country categories,0
Observed gender categories,0


,Raw,Prepared
Rows,10000,10000
Columns (including technical key),14,12
Modified values,0,0


## Split protocol fixed before exploration

Unique customers and no dates: a stratified random split assigns 60% to training, 20% to validation and 20% to testing, with seed 42. This assumes exchangeable customers; it is **not temporal validation**. Features might already reflect departure, and the split cannot remove that risk.

Stratification uses the class only to balance the partitions. Validation will select the model, calibration and strategy; the test remains sealed until the protocol is frozen in notebook 03. The technical ordering key is neither a predictor nor a date. Assignments are persisted and checked on every rerun. A change to the raw file or protocol stops execution rather than silently creating a new split.

In [4]:
from sklearn.model_selection import train_test_split
ids = clean.observation_id.to_numpy()
train_ids, rest_ids = train_test_split(ids, test_size=.4, stratify=clean.Exited, random_state=SEED)
val_ids, test_ids = train_test_split(rest_ids, test_size=.5, stratify=clean.set_index('observation_id').loc[rest_ids,'Exited'], random_state=SEED)
split = pd.DataFrame({'observation_id': ids, 'split': 'train'})
split.loc[split.observation_id.isin(val_ids),'split'] = 'validation'
split.loc[split.observation_id.isin(test_ids),'split'] = 'test'
assert len(set(train_ids) & set(val_ids)) == len(set(train_ids) & set(test_ids)) == len(set(val_ids) & set(test_ids)) == 0
assert len(train_ids)+len(val_ids)+len(test_ids) == len(df)
processed = ROOT / 'data/processed'
processed.mkdir(parents=True, exist_ok=True)
split_path = processed / 'splits.csv'
if split_path.exists():
 pd.testing.assert_frame_equal(pd.read_csv(split_path), split)
else:
 split.to_csv(split_path, index=False)
contract = {'raw_sha256': raw_hash, 'split_sha256': sha(split_path), 'seed': SEED, 'target': 'Exited', 'positive_class': 1, 'definition': 'fermeture de compte / départ, convention pédagogique documentée par concordance Microsoft Learn', 'operationally_validated': False, 'horizon': None, 'decision_date': None, 'source_documentation': 'https://learn.microsoft.com/en-us/fabric/data-science/customer-churn', 'source_access_date': '2026-11-09', 'row_unit': 'client unique dans le fichier', 'feature_timing_verified': False, 'quality': quality, 'excluded_predictors': ['RowNumber','CustomerId','Surname','observation_id']}
contract_path = processed / 'contract.json'
if contract_path.exists():
 assert json.loads(contract_path.read_text()) == contract, 'Contract changed: a new study is required'
else:
 contract_path.write_text(json.dumps(contract, ensure_ascii=False, indent=2))
clean.to_csv(processed / 'customers.csv', index=False)
assert sha(raw_path) == raw_hash
# Show split sizes without exploring test features or relationships.
display(split['split'].value_counts().rename('clients').to_frame())
print('Raw file preserved; SHA-256:',raw_hash)

Raw file preserved; SHA-256: 3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0


,Customers
split,
train,6000
validation,2000
test,2000


## Outcome and handoff

All 10,000 rows are retained, with no values corrected. Three identification columns are excluded and one technical key is added only for joins. No business identifier is displayed or passed downstream. `data/processed/customers.csv`, `splits.csv` and `contract.json` are local working files, not public exports.

Notebook 02 will analyze training rows only. Learned preprocessing belongs in notebook 03 pipelines. The educational definition supports comparisons of models for the supplied label; operational definition and chronology remain prerequisites for any campaign decision. The independent review is recorded in `reports/review.md`.